# Example 7.5 / 1: Switch a Capacity with else

Companion notebook for **7.5 Implication and Indicator Constraints** in the AMPL lab Documentation, section "Switch a Capacity with else". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Switch a Capacity with else

In the steel model, overtime adds 8 hours to the 40 available but costs 30,000. `Overtime = 1`
selects the larger limit; the `else` branch keeps the regular limit when it is 0:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;                         # products
param rate {PROD} > 0;            # tons produced per hour
param profit {PROD};              # profit per ton
param market {PROD} >= 0;         # most that can be sold, in tons
param avail >= 0;                 # regular hours available
param extra >= 0;                 # additional hours with overtime
param overtime_cost >= 0;         # fixed cost of working overtime

var Make {p in PROD} >= 0, <= market[p];  # tons produced
var Overtime binary;                       # 1 if overtime is worked

maximize NetProfit:
    sum {p in PROD} profit[p] * Make[p] - overtime_cost * Overtime;
# overtime raises the hour limit; otherwise the regular limit holds
subject to Time:
    Overtime = 1 ==> sum {p in PROD} (1 / rate[p]) * Make[p] <= avail + extra
               else  sum {p in PROD} (1 / rate[p]) * Make[p] <= avail;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils", "plate"]
ampl.param["rate"] = {"bands": 200, "coils": 140, "plate": 160}
ampl.param["profit"] = {"bands": 25, "coils": 30, "plate": 29}
ampl.param["market"] = {"bands": 6000, "coils": 4000, "plate": 3500}
ampl.param["avail"] = 40
ampl.param["extra"] = 8
ampl.param["overtime_cost"] = 30000

In [5]:
# run.py
ampl.solve(solver="highs")
print("Overtime =", round(ampl.var["Overtime"].value(), 6))
print("NetProfit =", ampl.obj["NetProfit"].value())
print(ampl.var["Make"].to_dict())

HiGHS 1.14.0: optimal solution; objective 203520
5 simplex iterations
1 branching nodes
Overtime = 1.0
NetProfit = 203520.0
{'bands': 6000, 'coils': 0, 'plate': 2880}


### Pure AMPL: data.dat and model.run

Without overtime the best profit is 196,400. The 8 extra hours make 1,280 more tons of plate,
worth 37,120, which exceeds the 30,000 cost, so the solver sets `Overtime = 1` (within the
solver's integrality tolerance, so the value is rounded for printing). HiGHS has no
implication constraints of its own: the MP interface converts the rule into linear rows with a
binary, using the bounds on `Make`, as described in
How MP Solvers Reformulate Your Model (AMPL lab documentation, section 8.6).

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
param: PROD: rate profit market :=
  bands  200  25  6000
  coils  140  30  4000
  plate  160  29  3500 ;
param avail := 40;
param extra := 8;
param overtime_cost := 30000;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve;
display Overtime, NetProfit, Make;

HiGHS 1.14.0: optimal solution; objective 203520
5 simplex iterations
1 branching nodes
Overtime = 1
NetProfit = 203520

Make [*] :=
bands  6000
coils     0
plate  2880
;

